In [15]:

import os
import numpy as np
from collections import Counter, defaultdict

print("All libraries imported successfully.")
print("NumPy version:", np.__version__)


All libraries imported successfully.
NumPy version: 2.1.3


In [16]:

def evaluate_tags(y_true, y_pred, labels=None):
    if len(y_true) != len(y_pred):
        raise ValueError("Actual and predicted tag counts do not match.")

    if labels is None:
        labels = ["B-NP", "I-NP", "O"]

    results = {}
    for tag in labels:
        tp = sum(a == tag and p == tag for a, p in zip(y_true, y_pred))
        fp = sum(a != tag and p == tag for a, p in zip(y_true, y_pred))
        fn = sum(a == tag and p != tag for a, p in zip(y_true, y_pred))

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        results[tag] = {
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1
        }

    accuracy = (
        sum(a == p for a, p in zip(y_true, y_pred)) / len(y_true)
        if y_true else 0.0
    )

    print(f"{'Tag':<10}{'Precision':>12}{'Recall':>12}{'F1-score':>12}")
    for tag, scores in results.items():
        print(
            f"{tag:<10}"
            f"{scores['Precision']:>12.4f}"
            f"{scores['Recall']:>12.4f}"
            f"{scores['F1-score']:>12.4f}"
        )

    print(f"\nToken Accuracy: {accuracy:.4f}")
    return results


In [17]:
# Cell 3: Mount Google Drive and load the datasets

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:



TRAIN_PATH = "/content/drive/MyDrive/Colab Notebooks/ILLM/Lab7-Data/train_NP.txt"
TEST_PATH  = "/content/drive/MyDrive/Colab Notebooks/ILLM/Lab7-Data/test_NP.txt"

def load_conll_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found: {file_path}\n"
            "Check the file location in Google Drive."
        )

    sentences = []
    sentence = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            # A blank line marks the end of a sentence
            if not line:
                if sentence:
                    sentences.append(sentence)
                    sentence = []
                continue

            parts = line.split()

            # Expected format: word ... tag
            if len(parts) >= 2:
                word = parts[0]
                tag = parts[-1]
                sentence.append((word, tag))

    if sentence:
        sentences.append(sentence)

    return sentences


train_data = load_conll_data(TRAIN_PATH)
test_data = load_conll_data(TEST_PATH)

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))
print("\nFirst training sentence:")
print(train_data[0] if train_data else "No training sentences found.")


Training sentences: 823
Testing sentences: 77

First training sentence:
[('Rockwell', 'B-NP'), ('International', 'I-NP'), ('Corp.', 'I-NP'), ("'s", 'B-NP'), ('Tulsa', 'I-NP'), ('unit', 'I-NP'), ('said', 'O'), ('it', 'B-NP'), ('signed', 'O'), ('a', 'B-NP'), ('tentative', 'I-NP'), ('agreement', 'I-NP'), ('extending', 'O'), ('its', 'B-NP'), ('contract', 'I-NP'), ('with', 'O'), ('Boeing', 'B-NP'), ('Co.', 'I-NP'), ('to', 'O'), ('provide', 'O'), ('structural', 'B-NP'), ('parts', 'I-NP'), ('for', 'O'), ('Boeing', 'B-NP'), ("'s", 'B-NP'), ('747', 'I-NP'), ('jetliners', 'I-NP'), ('.', 'O')]


In [19]:

# Cell 4: Check tags, empty sentences, and dataset sizes

STATES = ["B-NP", "I-NP", "O"]

def inspect_dataset(data, name):
    all_tags = [tag for sentence in data for word, tag in sentence]
    all_words = [word for sentence in data for word, tag in sentence]

    print(f"\n{name}")
    print("-" * 35)
    print("Sentences:", len(data))
    print("Total tokens:", len(all_words))
    print("Unique words:", len(set(all_words)))
    print("Tag counts:", Counter(all_tags))
    print("Unexpected tags:", sorted(set(all_tags) - set(STATES)))

    if not data:
        raise ValueError(f"{name} is empty.")

    if not all_tags:
        raise ValueError(f"{name} contains no tags.")


inspect_dataset(train_data, "Training Dataset")
inspect_dataset(test_data, "Testing Dataset")



Training Dataset
-----------------------------------
Sentences: 823
Total tokens: 19172
Unique words: 4570
Tag counts: Counter({'O': 8394, 'I-NP': 5727, 'B-NP': 5051})
Unexpected tags: []

Testing Dataset
-----------------------------------
Sentences: 77
Total tokens: 1896
Unique words: 744
Tag counts: Counter({'O': 842, 'I-NP': 579, 'B-NP': 475})
Unexpected tags: []


In [20]:

# Cell 5: Build HMM counts and probability tables

K = 0.1  # Add-K smoothing constant

initial_counts = Counter()
transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()
vocab = set()

# Count initial, transition, and emission frequencies
for sentence in train_data:
    if not sentence:
        continue

    # Count the first tag of each sentence
    initial_counts[sentence[0][1]] += 1

    previous_tag = None

    for word, tag in sentence:
        vocab.add(word)
        tag_counts[tag] += 1
        emission_counts[tag][word] += 1

        # Count transitions only within a sentence
        if previous_tag is not None:
            transition_counts[previous_tag][tag] += 1

        previous_tag = tag

# Number of hidden states and vocabulary size
num_states = len(STATES)
vocab.add("<UNK>")
vocab_size = len(vocab)

num_sentences = sum(bool(sentence) for sentence in train_data)

# 1. Initial probabilities with Add-K smoothing
initial_prob = {
    tag: (initial_counts[tag] + K) /
         (num_sentences + K * num_states)
    for tag in STATES
}

# 2. Transition probabilities with Add-K smoothing
transition_prob = {}

for previous_tag in STATES:
    # Count only transitions leaving this tag
    outgoing_count = sum(transition_counts[previous_tag].values())

    denominator = outgoing_count + K * num_states

    transition_prob[previous_tag] = {
        current_tag: (
            transition_counts[previous_tag][current_tag] + K
        ) / denominator
        for current_tag in STATES
    }

# Display results
print("Initial probabilities:")
for tag in STATES:
    print(f"{tag}: {initial_prob[tag]:.6f}")

print("\nTransition probabilities:")
for previous_tag in STATES:
    print(previous_tag, transition_prob[previous_tag])
    print(
        "Probability sum:",
        round(sum(transition_prob[previous_tag].values()), 6)
    )

print("\nVocabulary size:", vocab_size)
print("Tag counts:", tag_counts)

print("\nInitial probability sum:",
      round(sum(initial_prob.values()), 6))


Initial probabilities:
B-NP: 0.657233
I-NP: 0.000121
O: 0.342645

Transition probabilities:
B-NP {'B-NP': 0.029522998633744527, 'I-NP': 0.6956220422549155, 'O': 0.2748549591113399}
Probability sum: 1.0
I-NP {'B-NP': 0.04587541350883027, 'I-NP': 0.38753434967531897, 'O': 0.5665902368158507}
Probability sum: 1.0
O {'B-NP': 0.5403292777770455, 'I-NP': 1.3181656406944097e-05, 'O': 0.45965754056654756}
Probability sum: 1.0

Vocabulary size: 4571
Tag counts: Counter({'O': 8394, 'I-NP': 5727, 'B-NP': 5051})

Initial probability sum: 1.0


In [21]:

# Cell 6: Emission probabilities with Add-K smoothing

def get_emission_prob(tag, word):
    if word not in vocab:
        word = "<UNK>"

    denominator = tag_counts[tag] + K * vocab_size

    return (
        emission_counts[tag][word] + K
    ) / denominator


# Test an emission probability
sample_word = train_data[0][0][0]

print("Sample word:", sample_word)

for tag in STATES:
    print(
        f"P({sample_word} | {tag}) = "
        f"{get_emission_prob(tag, sample_word):.8f}"
    )


Sample word: Rockwell
P(Rockwell | B-NP) = 0.00074436
P(Rockwell | I-NP) = 0.00001617
P(Rockwell | O) = 0.00001130


In [22]:

# Cell 7: Forward algorithm in log space

def forward_log_probability(words):
    if not words:
        return 0.0

    # First word
    alpha = np.array([
        np.log(initial_prob[tag])
        + np.log(get_emission_prob(tag, words[0]))
        for tag in STATES
    ])

    # Remaining words
    for word in words[1:]:
        next_alpha = []

        for j, current_tag in enumerate(STATES):
            candidates = np.array([
                alpha[i]
                + np.log(transition_prob[previous_tag][current_tag])
                for i, previous_tag in enumerate(STATES)
            ])

            value = np.logaddexp.reduce(candidates)
            value += np.log(get_emission_prob(current_tag, word))
            next_alpha.append(value)

        alpha = np.array(next_alpha)

    # Sum over all possible final tags
    return float(np.logaddexp.reduce(alpha))


def forward_probability(words):
    log_probability = forward_log_probability(words)

    # The ordinary probability may underflow for long sentences
    probability = np.exp(log_probability) if log_probability > -745 else 0.0

    return probability, log_probability


print("Forward algorithm implemented successfully.")


Forward algorithm implemented successfully.


In [23]:

# Cell 8: Calculate probabilities for sample test sentences

num_samples = min(5, len(test_data))

for i in range(num_samples):
    words = [word for word, tag in test_data[i]]
    probability, log_probability = forward_probability(words)

    print(f"\nSentence {i + 1}:")
    print("Words:", " ".join(words))
    print("Log probability:", log_probability)
    print("Probability:", probability)



Sentence 1:
Words: Confidence in the pound is widely expected to take another sharp dive if trade figures for September , due for release tomorrow , fail to show a substantial improvement from July and August 's near-record deficits .
Log probability: -271.50872502840235
Probability: 1.2169113215237017e-118

Sentence 2:
Words: Chancellor of the Exchequer Nigel Lawson 's restated commitment to a firm monetary policy has helped to prevent a freefall in sterling over the past week .
Log probability: -197.99241508623118
Probability: 1.0303544932743322e-86

Sentence 3:
Words: But analysts reckon underlying support for sterling has been eroded by the chancellor 's failure to announce any new policy measures in his Mansion House speech last Thursday .
Log probability: -221.49612463285177
Probability: 6.389329846711681e-97

Sentence 4:
Words: This has increased the risk of the government being forced to increase base rates to 16 % from their current 15 % level to defend the pound , economists

In [24]:

# Cell 9: Viterbi decoding in log space

def viterbi_decode(words):
    if not words:
        return []

    # Initialization
    dp = np.array([
        np.log(initial_prob[tag])
        + np.log(get_emission_prob(tag, words[0]))
        for tag in STATES
    ])

    # Store the best previous state for each position and current state
    backpointers = []

    # Recursion
    for word in words[1:]:
        next_dp = np.zeros(len(STATES))
        backpointer = np.zeros(len(STATES), dtype=int)

        for j, current_tag in enumerate(STATES):
            candidates = np.array([
                dp[i]
                + np.log(transition_prob[previous_tag][current_tag])
                for i, previous_tag in enumerate(STATES)
            ])

            best_previous = int(np.argmax(candidates))
            backpointer[j] = best_previous

            next_dp[j] = (
                candidates[best_previous]
                + np.log(get_emission_prob(current_tag, word))
            )

        dp = next_dp
        backpointers.append(backpointer)

    # Find the best final state
    best_last_state = int(np.argmax(dp))

    # Backtrack to recover the best tag sequence
    best_path = [best_last_state]

    for backpointer in reversed(backpointers):
        best_path.append(int(backpointer[best_path[-1]]))

    best_path.reverse()

    return [STATES[state] for state in best_path]


print("Viterbi algorithm implemented successfully.")


Viterbi algorithm implemented successfully.


In [25]:

# Cell 10: Predict tags using Viterbi

y_true = []
y_pred = []
predicted_sentences = []

for sentence in test_data:
    words = [word for word, tag in sentence]
    actual_tags = [tag for word, tag in sentence]

    predicted_tags = viterbi_decode(words)

    if len(actual_tags) != len(predicted_tags):
        raise ValueError("Actual and predicted sequence lengths differ.")

    y_true.extend(actual_tags)
    y_pred.extend(predicted_tags)

    predicted_sentences.append({
        "words": words,
        "actual_tags": actual_tags,
        "predicted_tags": predicted_tags
    })


print("Total test tokens:", len(y_true))
print("Total predicted tokens:", len(y_pred))

print("\nSample prediction:")
if predicted_sentences:
    sample = predicted_sentences[0]

    for word, actual, predicted in zip(
        sample["words"],
        sample["actual_tags"],
        sample["predicted_tags"]
    ):
        print(f"{word:<20} Actual: {actual:<6} Predicted: {predicted}")


Total test tokens: 1896
Total predicted tokens: 1896

Sample prediction:
Confidence           Actual: B-NP   Predicted: B-NP
in                   Actual: O      Predicted: O
the                  Actual: B-NP   Predicted: B-NP
pound                Actual: I-NP   Predicted: I-NP
is                   Actual: O      Predicted: O
widely               Actual: O      Predicted: B-NP
expected             Actual: O      Predicted: O
to                   Actual: O      Predicted: O
take                 Actual: O      Predicted: O
another              Actual: B-NP   Predicted: B-NP
sharp                Actual: I-NP   Predicted: I-NP
dive                 Actual: I-NP   Predicted: I-NP
if                   Actual: O      Predicted: O
trade                Actual: B-NP   Predicted: B-NP
figures              Actual: I-NP   Predicted: I-NP
for                  Actual: O      Predicted: O
September            Actual: B-NP   Predicted: B-NP
,                    Actual: O      Predicted: O
due            

In [26]:

# Cell 11: Evaluate HMM predictions

results = evaluate_tags(
    y_true,
    y_pred,
    labels=STATES
)


Tag          Precision      Recall    F1-score
B-NP            0.7743      0.8737      0.8210
I-NP            0.8775      0.8411      0.8589
O               0.9106      0.8705      0.8901

Token Accuracy: 0.8623


In [27]:

# Cell 12: Final assignment summary

correct = sum(
    actual == predicted
    for actual, predicted in zip(y_true, y_pred)
)

total = len(y_true)
accuracy = correct / total if total else 0.0

print("=" * 45)
print("HMM NOUN PHRASE TAGGING — FINAL SUMMARY")
print("=" * 45)
print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))
print("Training vocabulary size:", vocab_size)
print("Smoothing method: Add-K")
print("K value:", K)
print("Hidden states:", STATES)
print("Test tokens:", total)
print("Correct predictions:", correct)
print(f"Token accuracy: {accuracy:.4f}")
print(f"Token accuracy (%): {accuracy * 100:.2f}%")
print("\nCompleted:")
print("1. Initial, transition, and emission probabilities")
print("2. Forward algorithm for sentence probability")
print("3. Viterbi algorithm for tag prediction")
print("4. Precision, recall, F1-score, and token accuracy")


HMM NOUN PHRASE TAGGING — FINAL SUMMARY
Training sentences: 823
Testing sentences: 77
Training vocabulary size: 4571
Smoothing method: Add-K
K value: 0.1
Hidden states: ['B-NP', 'I-NP', 'O']
Test tokens: 1896
Correct predictions: 1635
Token accuracy: 0.8623
Token accuracy (%): 86.23%

Completed:
1. Initial, transition, and emission probabilities
2. Forward algorithm for sentence probability
3. Viterbi algorithm for tag prediction
4. Precision, recall, F1-score, and token accuracy
